In [ ]:
import numpy as np
import torch
from src.ccpfn import CEPOEstimator 

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In this example, we use one of the semi synthetic datasets described in our paper - warfarin. You can refer to the README file for more information and for the necessary data to load into the directory.

In [ ]:
# Loading data

from src.ccpfn.benchmarks import SemiSyntheticEvalDataset
warfarin_path = "PATH_TO_WARFARIN_DATASET"  # Replace with the actual path to the Warfarin dataset

warfarin_dataset_cepo, _ = SemiSyntheticEvalDataset(
    csv_path=warfarin_path,
)

# Single CEPO Estimation

In [ ]:
estimator = CEPOEstimator(device=device)
estimator.fit(
    warfarin_dataset_cepo.X_train,
    warfarin_dataset_cepo.t_train,
    warfarin_dataset_cepo.y_train,
)

estimated_cepos = estimator.estimate_cepos(warfarin_dataset_cepo.X_test, warfarin_dataset_cepo.t_test)

# Dose Response Visualization

In [ ]:
import matplotlib.pyplot as plt

estimator = CEPOEstimator(device=device)
estimator.fit(
    warfarin_dataset_cepo.X_train,
    warfarin_dataset_cepo.t_train,
    warfarin_dataset_cepo.y_train,
)

# Choose one covariate set and evaluate its potential outcome over a grid of doses.
covariate_index = 0
x_fixed = warfarin_dataset_cepo.X_test[covariate_index : covariate_index + 1]

t_min = warfarin_dataset_cepo.t_train.min()
t_max = warfarin_dataset_cepo.t_train.max()
t_grid = np.linspace(t_min, t_max, 100)
X_grid = np.repeat(x_fixed, len(t_grid), axis=0)

estimated_response = estimator.estimate_cepo(X_grid, t_grid)

plt.figure(figsize=(7, 4))
plt.plot(t_grid, estimated_response, label="Estimated dose-response")
plt.scatter(
    warfarin_dataset_cepo.t_test[covariate_index],
    warfarin_dataset_cepo.true_cepo[covariate_index],
    color="black",
    zorder=3,
    label="Held-out CEPO target",
)
plt.xlabel("Dose")
plt.ylabel("Expected outcome")
plt.title(f"Dose-response for covariate set {covariate_index}")
plt.legend()
plt.tight_layout()
plt.show()